In [1]:
#makemore翻篇了！接下来是nanoGPT
#我们要正式开始进入Transformer了！

In [2]:
#!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
#这一步是下载数据集：tiny莎士比亚
#这一步在国内通常有些慢
#国内的读者可以用魔法直接打开网页后保存到目录里

In [3]:
#标准的解析文本写法，以只读方式用UTF-8编码打开文件为f,text就是超长的一个字符串 
with open('input.txt','r',encoding='utf-8') as f:
    text=f.read()

In [4]:
print(text[:1000])
print("\n总字符量:",len(text))

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [5]:
chars=sorted(list(set(text)))#这是我们用过好多次的提取字符的代码
vocab_size=len(chars)
for c in chars:
    print(repr(c),end="   ") # repr会把不可见字符转成\n这种可见写法
print("\n总字符量:",vocab_size)

'\n'   ' '   '!'   '$'   '&'   "'"   ','   '-'   '.'   '3'   ':'   ';'   '?'   'A'   'B'   'C'   'D'   'E'   'F'   'G'   'H'   'I'   'J'   'K'   'L'   'M'   'N'   'O'   'P'   'Q'   'R'   'S'   'T'   'U'   'V'   'W'   'X'   'Y'   'Z'   'a'   'b'   'c'   'd'   'e'   'f'   'g'   'h'   'i'   'j'   'k'   'l'   'm'   'n'   'o'   'p'   'q'   'r'   's'   't'   'u'   'v'   'w'   'x'   'y'   'z'   
总字符量: 65


In [6]:
stoi={ch:i for i,ch in enumerate(chars)}
itos={i:ch for i,ch in enumerate(chars)}
#还是构建字符表，但是现在我们可以构建函数来实现映射
encode=lambda s: [stoi[c] for c in s]
decode=lambda l: ''.join([itos[i] for i in l])

print(encode("Hello GPT!"))
print(decode(encode("Hello GPT!")))

[20, 43, 50, 50, 53, 1, 19, 28, 32, 2]
Hello GPT!


In [7]:
#我们用已经很熟悉的torch来包装text
import torch
data=torch.tensor(encode(text),dtype=torch.long)
print(data.shape)
print(data[:1000])

torch.Size([1115394])
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
        47, 59, 57

In [8]:
#区分训练集和验证集
n=int(0.9*len(text))
train_data=data[:n]
val_data=data[n+1:]

In [9]:
#接下来区分块
block_size=8
train_data[:block_size]

tensor([18, 47, 56, 57, 58,  1, 15, 47])

In [12]:
#在这里，我们会实现用前1～8个字符对应下一个字符
x=train_data[:block_size]
y=train_data[1:block_size+1]
for t in range(block_size):
    context=x[:t+1]
    target=y[t]
    print(f"context:{context},target:{target}")

context:tensor([18]),target:47
context:tensor([18, 47]),target:56
context:tensor([18, 47, 56]),target:57
context:tensor([18, 47, 56, 57]),target:58
context:tensor([18, 47, 56, 57, 58]),target:1
context:tensor([18, 47, 56, 57, 58,  1]),target:15
context:tensor([18, 47, 56, 57, 58,  1, 15]),target:47
context:tensor([18, 47, 56, 57, 58,  1, 15, 47]),target:58


In [13]:
#这里和之前的不一样了
#因为Transformer和之前的模型有一个不同点就是它不要求上下文是固定长度
#所以我们现在是可以用1～8个字符来预测下一个的

In [16]:
torch.manual_seed(1337)
batch_size=4#批大小
block_size=8#字符块大小


def get_batch(split):
    data=train_data if split=="train" else val_data
    ix=torch.randint(len(data)-block_size,(batch_size,))
    #torch.stack会用新维度把这些东西拼在一起
    #所以也就是对于x和y把4个形状为(8)的tensor拼成(4*8)的
    x=torch.stack([data[i:i+block_size]for i in ix])
    y=torch.stack([data[i+1:i+block_size+1]for i in ix])
    return x,y

#这里是展示我们的input和target
xb,yb=get_batch("train")
print("inputs:")
print(xb.shape)
print(xb)
print("target:")
print(yb.shape)
print(yb)

print('-----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
target:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
-----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [54]:
import torch
import torch.nn as nn#这个是torch的神经网络模块，我们之前手动实现的Linear等等都可以用这个实现
from torch.nn import functional as F
torch.manual_seed(1337)

#有的读者一开始看这里可能会不理解，其实我们就是开始用torch来代替我们的手写部分了
#还记得我们之前实现的embedding吗

# class Embedding:
#   def __init__(self, num_embeddings, embedding_dim):
#     self.weight = torch.randn((num_embeddings, embedding_dim))
#   def __call__(self, IX):
#     self.out = self.weight[IX]
#     return self.out
#   def parameters(self):
#     return [self.weight]

#这里几乎差不多
#构造就是一个vocab_size*vocab_size的矩阵
#调用就是一个矩阵乘法
#这里我们实现的是一个bigram
#所以我们直接用Embedding来当做我们的概率矩阵
#embedding就是个查表，所以把xb的全部元素都查了一下
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):#可以没有targets这样的话就可以直接用于生成
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets == None:
            loss=None
        else:
            #这几步是因为在pytorch内部，是按照BCT的顺序来排列的而不是BTC
            #所以我们需要调整一下
            #把X和Y压缩成一维的
            #这样就让BT全部压缩到B就符合条件了
            B,T,C=logits.shape
            logits=logits.view(B*T,C)
            targets=targets.view(-1)
            loss=F.cross_entropy(logits,targets)
            
        return logits,loss

    def generate(self, idx, max_new_tokens):#生成文字函数
        #这一步其实就是拿模型在生成新的字符
        #但是这次的生成方式是拼接
        #我们把新的字符直接在idx后面去拼接
        #这就非常像Transformer了
        #我们之前的生成的姓名是直接生成
        #现在是生成一个字之后再生成，再生成，全部拼成完整的一句话
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx
 

m = BigramLanguageModel(vocab_size)
logits,loss = m(xb, yb)#这一步是nn.Module的特性，相当于 m.forward(xb,yb)
#这一步就相当于xb去调用一下embedding
#embedding就是个查表，所以把xb的全部元素都查了一下
print(xb.shape)
print(yb.shape)
print(logits.shape)
print(loss)
print(decode(m.generate(torch.zeros((1,1),dtype=torch.long),max_new_tokens=100)[0].tolist()))

torch.Size([4, 8])
torch.Size([4, 8])
torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ
